# distill_safety — Colab launcher

Thin launcher for `distill_safety`. Teachers are cached under `$DRIVE_ROOT/outputs/teachers/<key>` and reused across all sweeps. Every stage writes checkpoints / shards / `.done` markers to Google Drive, so after a disconnect just **re-run all cells**.

Runtime → Change runtime type → **T4** (`configs/pilot_3b.yaml`) or **A100 / L4** (`configs/default_7b.yaml`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO_URL = 'https://github.com/AlbertStephen/distill_safety.git'
DRIVE_ROOT = '/content/drive/MyDrive/distill_safety'
import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
if not os.path.exists('/content/distill_safety'):
    !git clone -q $REPO_URL /content/distill_safety
else:
    !git -C /content/distill_safety pull -q
%cd /content/distill_safety
!pip install -q -r requirements-colab.txt
!pip uninstall -y -q torchao  # Colab preinstalled torchao 0.10 conflicts with peft
# Keep HF model cache on local SSD (Google Drive FUSE corrupts/slows safetensors mmap)
os.environ['HF_HOME'] = '/content/hf_cache'

In [ ]:
# Task: 'dealign' (jailbreak/de-alignment transfer) or 'backdoor'
TASK = 'dealign'
MODE = 'sweep'
CONFIG = 'configs/dealign_3b.yaml' if TASK == 'dealign' else 'configs/pilot_3b.yaml'
SWEEPS = ['ratio'] if TASK == 'dealign' else ['trigger', 'topic_natural']   # dealign: ratio|student_init|filter ; backdoor: dose|filter|temp|trigger|trigger_low|topic_natural
PREFIX = 'da3b_' if TASK == 'dealign' else 'p3b_'
EXTRA_OVERRIDES = ''

In [ ]:
sweep_script = 'scripts/run_dealign_sweep.py' if TASK == 'dealign' else 'scripts/run_sweep.py'
pipe_script = 'scripts/run_dealign_pipeline.py' if TASK == 'dealign' else 'scripts/run_pipeline.py'
if MODE == 'sweep':
    for sw in SWEEPS:
        print(f'\n==================== RUNNING {TASK.upper()} SWEEP: {sw} ====================')
        !python $sweep_script --config $CONFIG --sweep $sw --prefix $PREFIX output_root=$DRIVE_ROOT/outputs $EXTRA_OVERRIDES
else:
    !python $pipe_script --config $CONFIG output_root=$DRIVE_ROOT/outputs $EXTRA_OVERRIDES